In [2]:
import pandas as pd
import numpy as np
import pickle
import json
import matplotlib.pyplot as plt
from tueplots import bundles
bundles.icml2024()
import sys
sys.path.append("..")
from utils import visualize_response_matrix

benchmark = "lite"
scenario = "gsm"

with open(f"../data/gather_helm_data/results_with_z.pkl", "rb") as f:
    results = pickle.load(f)
results = results.loc[:, results.columns.get_level_values("benchmark") == benchmark]
results = results.loc[:, results.columns.get_level_values("scenario") == scenario]
results = results[~results.isna().all(axis=1)]

output_dir = "../result/gather_helm_data"
visualize_response_matrix(results, results, f"{output_dir}/response_matrix_{scenario}.png")
print(results.shape)
print(f"missing percentage: {results.isna().values.sum() / (results.shape[0] * results.shape[1])}")

(90, 997)
missing percentage: 0.0


# get all items

In [6]:
z_array = results.columns.get_level_values("z").astype(float).to_numpy()
sorted_indices = np.argsort(z_array)
df = results.iloc[:, sorted_indices]
col_names = [f"{i+1}" for i in range(len(z_array))]

new_columns = []
for col, col_name in zip(df.columns, col_names):
    new_columns.append(col + (col_name,))
df.columns = pd.MultiIndex.from_tuples(new_columns, names=["request.prompt", "references", "scenario", "benchmark", "z", "col_names"])
    
colname_to_z = {}
for col in df.columns:
    z = col[-2]
    col_name = col[-1]
    colname_to_z[col_name] = z
with open(f"colname2z_{scenario}_all.json", "w") as f:
    json.dump(colname_to_z, f, indent=4)

# write the item content to a txt
with open(f"item_content_{scenario}_all.txt", "w") as f:
    for col in df.columns:
        prompt = col[0]
        ref = col[1]
        z = col[-2]
        col_name = col[-1]
        f.write(f"###{col_name}, difficulty: {z}\n###prompt: {prompt}\n###reference: {ref}\n\n\n\n\n")

# save response matrix to csv
df.columns = df.columns.get_level_values("col_names")
df.index.name = None
df.to_csv(f"resmat_{scenario}_all.csv", index=False)

# Plot selected z distribution
plt.figure(figsize=(8, 4))
plt.hist(z_array, bins=30, alpha=0.7, color='purple', edgecolor='black')
plt.title("Distribution of Selected z")
plt.xlabel("z")
plt.savefig(f"z_distri_{scenario}_all.png", dpi=300, bbox_inches='tight')

# select hard & easy items

In [ ]:
num_easy = 10
num_hard = 7
total_item = num_easy + num_hard

# Step 1: Extract z as NumPy array
z_array = results.columns.get_level_values("z").astype(float).to_numpy()

# Step 2: Sort indices by z
sorted_indices = np.argsort(z_array)

# Step 3: Get easy and hard num_easy_hard indices
easy_idx = sorted_indices[-num_easy:]
hard_idx = sorted_indices[:num_hard]

# Step 4: Get MultiIndex columns
easy_cols = results.columns[easy_idx]
hard_cols = results.columns[hard_idx]

# Step 5: Combine and extract sub_df
selected_columns = easy_cols.append(hard_cols)
sub_df = results[selected_columns]

# Step 6: Rename columns
easy_names = [f"e{i+1}" for i in range(num_easy)]
difficult_names = [f"d{i+1}" for i in range(num_hard)]
diffeasy = easy_names + difficult_names

new_columns = []
for col, diffeasy in zip(sub_df.columns, diffeasy):
    new_columns.append(col + (diffeasy,))
sub_df.columns = pd.MultiIndex.from_tuples(new_columns, names=["request.prompt", "references", "scenario", "benchmark", "z", "diffeasy"])

# Save diffeasy_to_z to JSON file
diffeasy_to_z = {}
for col in sub_df.columns:
    z = col[-2]
    diffeasy = col[-1]
    diffeasy_to_z[diffeasy] = z
with open(f"colname2z_{scenario}_{total_item}.json", "w") as f:
    json.dump(diffeasy_to_z, f, indent=4)
    
# write the item content to a txt
with open(f"item_content_{scenario}_{total_item}.txt", "w") as f:
    for col in sub_df.columns:
        prompt = col[0]
        ref = col[1]
        z = col[-2]
        diffeasy = col[-1]
        f.write(f"###{diffeasy}, difficulty: {z}\n###prompt: {prompt}\n###reference: {ref}\n\n\n\n\n")

# save response matrix to csv
sub_df.columns = sub_df.columns.get_level_values("diffeasy")
sub_df.index.name = None
sub_df.to_csv(f"resmat_{scenario}_{total_item}.csv", index=False)

# Plot selected z distribution
selected_z_values = np.concatenate([z_array[hard_idx], z_array[easy_idx]])
plt.figure(figsize=(8, 4))
plt.hist(selected_z_values, bins=30, alpha=0.7, color='purple', edgecolor='black')
plt.title("Distribution of Selected z")
plt.xlabel("z")
plt.savefig(f"z_distri_{scenario}_{total_item}.png", dpi=300, bbox_inches='tight')